In [ ]:
import sys, pathlib
project_root = pathlib.Path.cwd().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from backtesting import Backtest

from trader.l1_data import fetch_h4
from trader.l2_features import build_bt_df
from trader.l4_signal_model import ConfluenceStrategy
from trader.backtest_harness import run_fold, walk_forward

In [ ]:
us30_raw = build_bt_df("^DJI")
gold_raw = build_bt_df("GC=F")

common_start = max(us30_raw.index[0], gold_raw.index[0])
common_end = min(us30_raw.index[-1], gold_raw.index[-1])

us30_bt_df = us30_raw.loc[common_start:common_end]
gold_bt_df = gold_raw.loc[common_start:common_end]

print("Common window:", common_start, "->", common_end)
print("US30 bars:", len(us30_bt_df), " Gold bars:", len(gold_bt_df))

In [ ]:
print("=== US30 (half-capital) ===")
us30_folds = walk_forward(us30_bt_df, cash=25_000)

print("\n=== Gold (half-capital) ===")
gold_folds = walk_forward(gold_bt_df, cash=25_000)

In [ ]:
def compound_equity(fold_results, starting_cash=25_000):
    fold_df = pd.DataFrame(fold_results)
    equity = starting_cash
    equities = []
    for r in fold_df["return_pct"]:
        equity *= (1 + r / 100)
        equities.append(equity)
    fold_df["equity_after_fold"] = equities
    return fold_df

us30_df = compound_equity(us30_folds)
gold_df = compound_equity(gold_folds)

n_folds = min(len(us30_df), len(gold_df))
us30_df = us30_df.iloc[:n_folds].reset_index(drop=True)
gold_df = gold_df.iloc[:n_folds].reset_index(drop=True)

portfolio = pd.DataFrame({
    "fold": range(1, n_folds + 1),
    "us30_test_end": us30_df["test_end"],
    "gold_test_end": gold_df["test_end"],
    "us30_equity": us30_df["equity_after_fold"],
    "gold_equity": gold_df["equity_after_fold"],
})
portfolio["portfolio_equity"] = portfolio["us30_equity"] + portfolio["gold_equity"]
print(portfolio)

plt.figure(figsize=(10, 5))
plt.plot(portfolio["fold"], portfolio["us30_equity"], marker="o", label="US30 only ($25k)")
plt.plot(portfolio["fold"], portfolio["gold_equity"], marker="o", label="Gold only ($25k)")
plt.plot(portfolio["fold"], portfolio["portfolio_equity"], marker="o", linewidth=2, color="black", label="Combined portfolio")
plt.axhline(50_000, color="gray", linestyle="--")
plt.title("US30 + Gold combined portfolio (half capital each)")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
fold_params = {'adx_threshold': 15, 'swing_lookback': 20, 'atr_sl_mult': 1.0, 'atr_tp_mult': 2.5}

test3 = us30_bt_df.loc["2025-08-26":"2025-11-26"]
bt3 = Backtest(test3, ConfluenceStrategy, cash=25_000, commission=0.0002, margin=1/30)
stats3 = bt3.run(**fold_params)
print("=== Fold 3 (Aug-Nov 2025) trades ===")
print(stats3._trades[["Size", "EntryTime", "ExitTime", "EntryPrice", "ExitPrice", "PnL", "ReturnPct"]])

test4 = us30_bt_df.loc["2025-11-26":"2026-02-26"]
bt4 = Backtest(test4, ConfluenceStrategy, cash=25_000, commission=0.0002, margin=1/30)
stats4 = bt4.run(**fold_params)
print("\n=== Fold 4 (Nov 2025-Feb 2026) trades ===")
print(stats4._trades[["Size", "EntryTime", "ExitTime", "EntryPrice", "ExitPrice", "PnL", "ReturnPct"]])

In [ ]:
for sl_mult in [1.0, 1.5, 2.0, 2.5]:
    params = {'adx_threshold': 15, 'swing_lookback': 20, 'atr_sl_mult': sl_mult, 'atr_tp_mult': 2.5}
    bt = Backtest(test3, ConfluenceStrategy, cash=25_000, commission=0.0002, margin=1/30)
    stats = bt.run(**params)
    print(f"Fold3 sl_mult={sl_mult}: Return={stats['Return [%]']:.2f}%  WinRate={stats['Win Rate [%]']:.1f}%  Trades={stats['# Trades']}")

for sl_mult in [1.0, 1.5, 2.0, 2.5]:
    params = {'adx_threshold': 15, 'swing_lookback': 20, 'atr_sl_mult': sl_mult, 'atr_tp_mult': 2.5}
    bt = Backtest(test4, ConfluenceStrategy, cash=25_000, commission=0.0002, margin=1/30)
    stats = bt.run(**params)
    print(f"Fold4 sl_mult={sl_mult}: Return={stats['Return [%]']:.2f}%  WinRate={stats['Win Rate [%]']:.1f}%  Trades={stats['# Trades']}")